# 🎬 IMDb Sentiment Analysis — Traditional NLP
Classify movie reviews as **Positive** or **Negative** using TF-IDF + Logistic Regression.

## 1. Install & Import

In [ ]:
import re
import string
import nltk
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

STOP_WORDS = set(stopwords.words('english'))
print('All imports done ✅')

## 2. Load Dataset

In [ ]:
# Downloads ~84 MB on first run, then cached automatically
train_df = pd.DataFrame(load_dataset('imdb', split='train'))
test_df  = pd.DataFrame(load_dataset('imdb', split='test'))

train_df['sentiment'] = train_df['label'].map({0: 'negative', 1: 'positive'})
test_df['sentiment']  = test_df['label'].map({0: 'negative', 1: 'positive'})

print(f'Train samples : {len(train_df):,}')
print(f'Test  samples : {len(test_df):,}')
train_df.head(3)

## 3. Explore the Data

In [ ]:
# Class distribution
counts = train_df['sentiment'].value_counts()
print('Class distribution:')
print(counts.to_string())

counts.plot(kind='bar', color=['#e74c3c','#2ecc71'], edgecolor='black', figsize=(5,3))
plt.title('Class Distribution (Train)')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 5 positive and 5 negative sample reviews
print('=== 5 POSITIVE REVIEWS ===')
for text in train_df[train_df['label']==1]['text'].head(5):
    print('-', text[:150].replace('\n',' '), '...\n')

print('=== 5 NEGATIVE REVIEWS ===')
for text in train_df[train_df['label']==0]['text'].head(5):
    print('-', text[:150].replace('\n',' '), '...\n')

In [ ]:
# Average review length
train_df['word_count'] = train_df['text'].str.split().str.len()
print(f"Average review length : {train_df['word_count'].mean():.0f} words")
print(f"Median review length  : {train_df['word_count'].median():.0f} words")

train_df['word_count'].clip(upper=800).hist(bins=40, color='steelblue', edgecolor='white', figsize=(7,3))
plt.title('Review Length Distribution')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

## 4. Text Preprocessing

In [ ]:
def clean_text(text):
    """Lowercase → strip HTML → remove punctuation → remove stopwords"""
    text = text.lower()                                      # lowercase
    text = re.sub(r'<[^>]+>', ' ', text)                    # remove HTML tags
    text = text.translate(str.maketrans('','', string.punctuation))  # remove punctuation
    text = re.sub(r'\d+', '', text)                         # remove digits
    tokens = word_tokenize(text)                             # tokenize
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]  # remove stopwords
    return ' '.join(tokens)

# Show before / after
sample = train_df['text'].iloc[0]
print('BEFORE:', sample[:200], '...\n')
print('AFTER :', clean_text(sample)[:200])

In [ ]:
print('Cleaning training reviews...')
X_train = train_df['text'].apply(clean_text)
print('Cleaning test reviews...')
X_test  = test_df['text'].apply(clean_text)

y_train = train_df['label'].values
y_test  = test_df['label'].values

print('Done ✅')

## 5. TF-IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),   # unigrams + bigrams
    max_features=50_000,
    min_df=2,
    sublinear_tf=True,    # log scaling
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Vocabulary size : {len(tfidf.vocabulary_):,}')
print(f'Train matrix    : {X_train_tfidf.shape}')
print(f'Test  matrix    : {X_test_tfidf.shape}')

## 6. Train — Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000, C=1.0)
model.fit(X_train_tfidf, y_train)
print('Training complete ✅')

## 7. Evaluate

In [ ]:
y_pred = model.predict(X_test_tfidf)

acc          = accuracy_score(y_test, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')

print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1-Score  : {f1:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Negative','Positive']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()

## 8. Try It Yourself

In [ ]:
def predict(review):
    cleaned = clean_text(review)
    vector  = tfidf.transform([cleaned])
    result  = model.predict(vector)[0]
    prob    = model.predict_proba(vector)[0][result]
    label   = '✅ POSITIVE' if result == 1 else '❌ NEGATIVE'
    print(f'{label}  (confidence: {prob:.2%})')

predict('This movie was absolutely amazing, I loved every second of it!')
predict('Terrible film. Boring plot, bad acting. Total waste of time.')
predict('It was okay, nothing special but not bad either.')